# 🛡️ SENTINELAI V16 — WebRTC ULTRA LOW LATENCY

### Architecture Professionnelle WebRTC
Les versions précédentes (V14/V15) utilisaient MJPEG-over-HTTP à travers un tunnel (Pinggy/Cloudflare). Ces tunnels sont des relais TCP génériques qui mettent la vidéo en mémoire tampon → **lag inévitable**.

### V16 : La méthode des pros (Zoom, Google Meet, Twitch)
- **WebRTC** transporte la vidéo en **UDP** (pas de retransmission bloquante comme TCP)
- Le tunnel ne sert plus qu'à un minuscule échange JSON de signalisation (~500 octets)
- La vidéo voyage **directement** entre Colab et votre navigateur via TURN/STUN
- Latence cible : **200-500ms** au lieu de 3-10 secondes
---
> ⚠️ `Exécution` → `Modifier le type d'exécution` → **T4 GPU**

In [ ]:
import warnings, os, logging
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

!pip install -q ultralytics supervision opencv-python-headless requests flask flask-cors huggingface_hub pysocks aiortc aiohttp av 2>/dev/null

import torch
print('═' * 60)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'✅ GPU : {props.name}')
else:
    print('❌ PAS DE GPU !')
print('═' * 60)

In [ ]:
# ⚠️ METTEZ ICI LE LIEN (PINGGY OU CLOUDFLARE) DE VOTRE CAMERA
STREAM_URL = 'https://obdmp-154-110-11-90.free.pinggy.net/videofeed'

TELEGRAM_BOT_TOKEN = '8878297698:AAHVye2ZHD_ZdZv8Ul8frmrCnmsJ2XVgFXA'
TELEGRAM_CHAT_ID   = '8102614326'
SMTP_USER   = 'firasmrabet1603@gmail.com'
SMTP_PASS   = 'nzlqovjjkjmuhjxb'
ALERT_EMAIL = 'firasmrabet1603@gmail.com'

CONF_PERSON   = 0.50
CONF_VIOLENCE = 0.85
CONF_WEAPON   = 0.80

FRAMES_VIOLENCE = 12
FRAMES_WEAPON   = 12
COOLDOWN_ALERTS = 120
IMGSZ = 416

HARMLESS_COCO = {39:'bottle',40:'wine glass',41:'cup',42:'fork',43:'knife',44:'spoon',45:'bowl',46:'banana',47:'apple',48:'sandwich',49:'orange',63:'laptop',64:'mouse',65:'remote',66:'keyboard',67:'cell phone',73:'book',74:'clock',75:'vase',76:'scissors',77:'teddy bear',78:'hair drier',79:'toothbrush',24:'backpack',25:'umbrella',26:'handbag',27:'tie',28:'suitcase'}
print('✅ Configuration chargée')

---
## 📥 Cellule 3 — Téléchargement Auto (API HuggingFace Hub)

In [ ]:
import os, shutil
from google.colab import drive
from huggingface_hub import hf_hub_download

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/SentinelAI_V5'
os.makedirs(DRIVE_DIR, exist_ok=True)

for m in ['violence_model.pt', 'weapon_model.pt']:
    dp, lp = os.path.join(DRIVE_DIR, m), os.path.join('/content', m)
    if os.path.exists(dp) and os.path.getsize(dp) < 1000000:
        os.remove(dp)
    if os.path.exists(lp) and os.path.getsize(lp) < 1000000:
        os.remove(lp)

print('⏳ Connexion sécurisée à HuggingFace...')
models_config = {
    'violence_model.pt': {'repo': 'Musawer14/fight_detection_yolov8', 'file': 'yolo_small_weights.pt'},
    'weapon_model.pt': {'repo': 'Subh775/Threat-Detection-YOLOv8n', 'file': 'weights/best.pt'}
}

for m_name, info in models_config.items():
    dp = os.path.join(DRIVE_DIR, m_name)
    lp = os.path.join('/content', m_name)
    
    if not os.path.exists(dp):
        print(f'📥 Téléchargement API : {m_name} depuis {info["repo"]} ...')
        cache_path = hf_hub_download(repo_id=info['repo'], filename=info['file'])
        shutil.copy2(cache_path, dp)
        print(f'💾 Modèle sauvegardé avec succès dans le Drive ({os.path.getsize(dp) // 1048576} Mo)')
        
    if not os.path.exists(lp):
        shutil.copy2(dp, lp)

print('✅ Modèles Haute Précision chargés et prêts !')

In [ ]:
import os, logging, warnings
import numpy as np
from ultralytics import YOLO
import torch
warnings.filterwarnings('ignore')
logging.getLogger('ultralytics').setLevel(logging.ERROR)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model_person = YOLO('yolov8n.pt')
model_violence = YOLO('/content/violence_model.pt') if os.path.exists('/content/violence_model.pt') else None
model_weapon = YOLO('/content/weapon_model.pt') if os.path.exists('/content/weapon_model.pt') else None

print('\n🔥 Compilation GPU...')
dummy = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
for _ in range(3): _ = model_person(dummy, verbose=False, device=DEVICE)
if model_violence: 
    print(f"📌 Classes Violence détectables : {model_violence.names}")
    for _ in range(3): _ = model_violence(dummy, verbose=False, device=DEVICE)
if model_weapon:
    print(f"📌 Classes Armes détectables : {model_weapon.names}")
    for _ in range(3): _ = model_weapon(dummy, verbose=False, device=DEVICE)
print('✅ GPU prêt !')

---
## 🔧 Cellule 5 — Classes Utilitaires

In [ ]:
import cv2, os, time, threading, requests, smtplib, gc
import numpy as np
from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from queue import Queue

# ── THREAD 1: LECTEUR VIDEO SNAPSHOT POLLING (Anti-lag tunnel) ──
class NoLagCamera:
    def __init__(self, url):
        # Convertir /videofeed ou /video en /shot.jpg pour du polling image par image
        self.url = url.replace('/videofeed', '/shot.jpg').replace('/video', '/shot.jpg')
        self.frame = None
        self.frame_id = 0
        self.running = True
        self.lock = threading.Lock()
        self.new_frame_event = threading.Event()
        threading.Thread(target=self._stream_reader, daemon=True).start()

    def _stream_reader(self):
        session = requests.Session()
        while self.running:
            try:
                r = session.get(self.url, timeout=2)
                if r.status_code == 200:
                    img_array = np.frombuffer(r.content, dtype=np.uint8)
                    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
                    if img is not None:
                        with self.lock:
                            self.frame = img
                            self.frame_id += 1
                        self.new_frame_event.set()
            except:
                pass
            time.sleep(0.05)

    def get_new_frame(self):
        if self.new_frame_event.wait(timeout=1.0):
            self.new_frame_event.clear()
            with self.lock:
                return True, self.frame.copy(), self.frame_id
        return False, None, -1

    def stop(self):
        self.running = False

# ── ALERTES ASYNCHRONES ──
alert_queue = Queue()
def alert_worker():
    while True:
        alert_type, frame = alert_queue.get()
        try:
            _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
            img_bytes = buf.tobytes()

            # Telegram
            try:
                url = f'https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendPhoto'
                ts = datetime.now().strftime('%H:%M:%S %d/%m/%Y')
                requests.post(url, data={'chat_id': TELEGRAM_CHAT_ID, 'caption': f'🚨 ALERTE {alert_type} — {ts}'},
                             files={'photo': ('alert.jpg', img_bytes, 'image/jpeg')}, timeout=10)
            except: pass

            # Email
            try:
                msg = MIMEMultipart()
                msg['From'], msg['To'] = SMTP_USER, ALERT_EMAIL
                msg['Subject'] = f'🚨 ALERTE {alert_type} — SentinelAI'
                ts = datetime.now().strftime('%H:%M:%S %d/%m/%Y')
                msg.attach(MIMEText(f'Alerte {alert_type} détectée à {ts}', 'plain'))
                att = MIMEBase('application', 'octet-stream')
                att.set_payload(img_bytes)
                encoders.encode_base64(att)
                att.add_header('Content-Disposition', 'attachment', filename='alert.jpg')
                msg.attach(att)
                with smtplib.SMTP_SSL('smtp.gmail.com', 465, timeout=10) as s:
                    s.login(SMTP_USER, SMTP_PASS)
                    s.send_message(msg)
            except: pass
        except: pass

threading.Thread(target=alert_worker, daemon=True).start()

# ── FILTRE TEMPOREL ──
class TemporalFilter:
    def __init__(self):
        self.counters = {}
        self.cooldowns = {}
    def update(self, key, detected, threshold, cooldown):
        now = time.time()
        if detected:
            self.counters[key] = self.counters.get(key, 0) + 1
        else:
            self.counters[key] = max(0, self.counters.get(key, 0) - 2)
        if self.counters.get(key, 0) >= threshold:
            if now - self.cooldowns.get(key, 0) > cooldown:
                self.cooldowns[key] = now
                self.counters[key] = 0
                return True
        return False

import supervision as sv

# ── MOTEUR IA ──
class SentinelEngine:
    def __init__(self):
        self.tracker = sv.ByteTrack(minimum_matching_threshold=0.6)
        self.person_box = sv.BoxAnnotator(thickness=2)
        self.person_label = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)
        self.danger_box = sv.BoxAnnotator(color=sv.Color.RED, thickness=3)
        self.danger_label = sv.LabelAnnotator(color=sv.Color.RED, text_scale=0.6, text_thickness=2)
        self.temporal = TemporalFilter()
        self.frame_counter = 0
        self.last_v_dets = None
        self.last_v_lbls = []
        self.last_w_dets = None
        self.last_w_lbls = []
        self.safe_keywords = {'laptop','cell phone','remote','keyboard','mouse','bottle','cup','phone','book','monitor','screen','tv','clock','backpack','bag','umbrella','suitcase','chair','table','desk'}

    def _is_truly_weapon(self, frame, x1, y1, x2, y2, h, w):
        bw, bh = x2-x1, y2-y1
        area_ratio = (bw * bh) / (w * h)
        if area_ratio > 0.25 or area_ratio < 0.001: return False
        aspect = bw / max(bh, 1)
        if aspect > 6 or aspect < 0.15: return False
        return True

    def process(self, frame):
        self.frame_counter += 1
        h, w = frame.shape[:2]
        annotated = frame.copy()
        alerts = []

        results = model_person(frame, imgsz=IMGSZ, conf=CONF_PERSON, classes=[0], verbose=False, device=DEVICE)
        dets = sv.Detections.from_ultralytics(results[0])
        dets = self.tracker.update_with_detections(dets)

        persons = []
        if len(dets) > 0:
            for i in range(len(dets)):
                x1,y1,x2,y2 = map(int, dets.xyxy[i])
                persons.append({'bbox': (x1,y1,x2,y2)})
            labels = [f'#{tid}' for tid in (dets.tracker_id if dets.tracker_id is not None else range(len(dets)))]
            annotated = self.person_box.annotate(scene=annotated, detections=dets)
            annotated = self.person_label.annotate(scene=annotated, detections=dets, labels=labels)

        run_secondary = (self.frame_counter % 5 == 0) and len(persons) > 0

        if run_secondary and model_violence:
            v_results = model_violence(frame, imgsz=IMGSZ, conf=CONF_VIOLENCE, verbose=False, device=DEVICE)
            v_dets = sv.Detections.from_ultralytics(v_results[0])
            if len(v_dets) > 0:
                valid_v = []
                v_labels = []
                for i in range(len(v_dets)):
                    vx1,vy1,vx2,vy2 = map(int, v_dets.xyxy[i])
                    near = any(p['bbox'][0]-50<(vx1+vx2)/2<p['bbox'][2]+50 and p['bbox'][1]-50<(vy1+vy2)/2<p['bbox'][3]+50 for p in persons)
                    if near:
                        valid_v.append(i)
                        cls_name = model_violence.names[int(v_dets.class_id[i])]
                        v_labels.append(f'⚠ {cls_name}')
                self.last_v_dets = v_dets[valid_v] if valid_v else None
                self.last_v_lbls = v_labels
                if self.temporal.update('violence', len(valid_v)>0, FRAMES_VIOLENCE, COOLDOWN_ALERTS):
                    alert_queue.put(('VIOLENCE', annotated.copy()))
                    alerts.append('VIOLENCE')
            else:
                self.last_v_dets = None
                self.temporal.update('violence', False, 1, 0)

        if run_secondary and model_weapon:
            w_results = model_weapon(frame, imgsz=IMGSZ, conf=CONF_WEAPON, verbose=False, device=DEVICE)
            w_dets = sv.Detections.from_ultralytics(w_results[0])
            if len(w_dets) > 0:
                valid_w = []
                w_labels = []
                for i in range(len(w_dets)):
                    cls_id = int(w_dets.class_id[i])
                    cls_name = model_weapon.names[cls_id].lower()
                    if any(safe_kw in cls_name for safe_kw in self.safe_keywords):
                        continue
                    wx1,wy1,wx2,wy2 = map(int, w_dets.xyxy[i])
                    wcx, wcy = (wx1+wx2)/2, (wy1+wy2)/2
                    near = any(p['bbox'][0]-80<wcx<p['bbox'][2]+80 and p['bbox'][1]-80<wcy<p['bbox'][3]+80 for p in persons)
                    if not near or not self._is_truly_weapon(frame, wx1, wy1, wx2, wy2, h, w): continue
                    valid_w.append(i)
                    w_labels.append(f'⚠ {cls_name.upper()}')
                self.last_w_dets = w_dets[valid_w] if valid_w else None
                self.last_w_lbls = w_labels
                if self.temporal.update('weapon', len(valid_w)>0, FRAMES_WEAPON, COOLDOWN_ALERTS):
                    alert_queue.put(('WEAPON', annotated.copy()))
                    alerts.append('ARME')
            else:
                self.last_w_dets = None
                self.temporal.update('weapon', False, 1, 0)

        if self.last_v_dets is not None and len(self.last_v_dets) > 0:
            annotated = self.danger_box.annotate(scene=annotated, detections=self.last_v_dets)
            annotated = self.danger_label.annotate(scene=annotated, detections=self.last_v_dets, labels=self.last_v_lbls)
        if self.last_w_dets is not None and len(self.last_w_dets) > 0:
            annotated = self.danger_box.annotate(scene=annotated, detections=self.last_w_dets)
            annotated = self.danger_label.annotate(scene=annotated, detections=self.last_w_dets, labels=self.last_w_lbls)

        if alerts:
            overlay = annotated.copy()
            cv2.rectangle(overlay, (0,0), (w,70), (0,0,200), -1)
            annotated = cv2.addWeighted(overlay, 0.6, annotated, 0.4, 0)
            cv2.putText(annotated, f'!! {" + ".join(alerts)} !!', (20,50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)

        return annotated

engine = SentinelEngine()

# ── THREAD 2: INFERENCE WORKER ──
class InferenceWorker:
    def __init__(self, cam, engine):
        self.cam = cam
        self.engine = engine
        self.latest_annotated = None
        self.latest_id = -1
        self.lock = threading.Lock()
        self.new_annotated_event = threading.Event()
        self.running = True
        self.fps = 0
        self.frame_count = 0
        threading.Thread(target=self._run, daemon=True).start()

    def _run(self):
        fps_start = time.time()
        fps_count = 0
        last_gc = time.time()
        while self.running:
            ret, frame, frame_id = self.cam.get_new_frame()
            if not ret or frame is None:
                continue
            
            annotated = self.engine.process(frame)
            
            fps_count += 1
            elapsed = time.time() - fps_start
            if elapsed >= 1.0:
                self.fps = fps_count / elapsed
                fps_count, fps_start = 0, time.time()
            
            h, w = annotated.shape[:2]
            cv2.rectangle(annotated, (0,h-40), (w,h), (0,0,0), -1)
            cv2.putText(annotated, 'SENTINELAI V16 (WebRTC)', (10,h-12), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,255), 2)
            cv2.putText(annotated, f'AI FPS: {int(self.fps)}', (w-130,h-12), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,0), 2)
            
            with self.lock:
                self.latest_annotated = annotated
                self.latest_id = frame_id
            self.new_annotated_event.set()
            
            self.frame_count += 1
            
            if time.time() - last_gc > 30:
                gc.collect()
                torch.cuda.empty_cache()
                last_gc = time.time()

    def get_latest_annotated(self):
        if self.new_annotated_event.wait(timeout=1.0):
            self.new_annotated_event.clear()
            with self.lock:
                return True, self.latest_annotated.copy()
        return False, None

print('✅ Moteur IA + InferenceWorker prêts')

---
## 🎥 Cellule 7 — WebRTC Server + Tunnel (Ultra Low Latency)

In [ ]:
import subprocess, re, asyncio, fractions
from aiohttp import web
from aiortc import RTCPeerConnection, RTCSessionDescription, VideoStreamTrack, RTCConfiguration, RTCIceServer
from av import VideoFrame
import logging
logging.getLogger('aiohttp').setLevel(logging.ERROR)
logging.getLogger('aiortc').setLevel(logging.ERROR)

# ── Nettoyage du port 5000 ──
import os
os.system('fuser -k 5000/tcp')
time.sleep(1)

# ── Démarrage Caméra + InferenceWorker ──
print(f'📡 Connexion à la Caméra: {STREAM_URL}')
cam = NoLagCamera(STREAM_URL)
time.sleep(2)
worker = InferenceWorker(cam, engine)
print('✅ InferenceWorker démarré (Thread 2)')

# ══════════════════════════════════════════════════════════════
# ═══ WebRTC VIDEO TRACK — Pousse les frames annotées en UDP ═══
# ══════════════════════════════════════════════════════════════
class AIVideoTrack(VideoStreamTrack):
    kind = 'video'
    def __init__(self, worker):
        super().__init__()
        self.worker = worker
        self._start = time.time()

    async def recv(self):
        pts, time_base = await self.next_timestamp()
        
        # Récupérer la dernière frame annotée par l'IA (thread-safe)
        ret, frame = self.worker.get_latest_annotated()
        if not ret or frame is None:
            # Frame noire si pas encore prêt
            frame = np.zeros((480, 640, 3), dtype=np.uint8)
            cv2.putText(frame, 'En attente de la camera...', (50, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        
        # Convertir OpenCV BGR → WebRTC VideoFrame
        vframe = VideoFrame.from_ndarray(frame, format='bgr24')
        vframe.pts = pts
        vframe.time_base = time_base
        return vframe

# ══════════════════════════════════════════════════════════════
# ═══ SERVEUR SIGNALISATION (aiohttp) — Échange SDP via HTTP ═══
# ══════════════════════════════════════════════════════════════
pcs = set()

# STUN + TURN publics pour traverser le NAT de Colab
ice_servers = [
    RTCIceServer(urls='stun:stun.l.google.com:19302'),
    RTCIceServer(urls='stun:stun1.l.google.com:19302'),
    RTCIceServer(
        urls=['turn:openrelay.metered.ca:80', 'turn:openrelay.metered.ca:443', 'turn:openrelay.metered.ca:443?transport=tcp'],
        username='openrelayproject',
        credential='openrelayproject'
    ),
]

async def handle_offer(request):
    """Reçoit l'offre SDP du navigateur, crée la réponse WebRTC."""
    params = await request.json()
    offer = RTCSessionDescription(sdp=params['sdp'], type=params['type'])
    
    pc = RTCPeerConnection(configuration=RTCConfiguration(iceServers=ice_servers))
    pcs.add(pc)
    
    @pc.on('connectionstatechange')
    async def on_state():
        state = pc.connectionState
        print(f'  📶 WebRTC: {state}')
        if state in ('failed', 'closed'):
            await pc.close()
            pcs.discard(pc)
    
    # Attacher le flux vidéo IA
    pc.addTrack(AIVideoTrack(worker))
    
    await pc.setRemoteDescription(offer)
    answer = await pc.createAnswer()
    await pc.setLocalDescription(answer)
    
    return web.json_response({
        'sdp': pc.localDescription.sdp,
        'type': pc.localDescription.type
    })

async def handle_health(request):
    return web.json_response({
        'status': 'ok',
        'fps': int(worker.fps),
        'frames': worker.frame_count,
        'peers': len(pcs),
        'version': 'V16-WebRTC'
    })

async def on_shutdown(app):
    coros = [pc.close() for pc in pcs]
    await asyncio.gather(*coros)
    pcs.clear()

# CORS middleware
@web.middleware
async def cors_middleware(request, handler):
    if request.method == 'OPTIONS':
        return web.Response(headers={
            'Access-Control-Allow-Origin': '*',
            'Access-Control-Allow-Methods': 'POST, GET, OPTIONS',
            'Access-Control-Allow-Headers': 'Content-Type',
        })
    resp = await handler(request)
    resp.headers['Access-Control-Allow-Origin'] = '*'
    return resp

app = web.Application(middlewares=[cors_middleware])
app.router.add_post('/offer', handle_offer)
app.router.add_get('/health', handle_health)
app.router.add_options('/offer', lambda r: web.Response())
app.on_shutdown.append(on_shutdown)

# Lancer aiohttp dans un thread séparé de manière propre (sans signaux)
def run_webrtc_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    runner = web.AppRunner(app, handle_signals=False)
    loop.run_until_complete(runner.setup())
    site = web.TCPSite(runner, '0.0.0.0', 5001)
    loop.run_until_complete(site.start())
    loop.run_forever()

threading.Thread(target=run_webrtc_server, daemon=True).start()
print('✅ Serveur WebRTC (aiortc) démarré sur :5001/offer')

# ══════════════════════════════════════════════════════════════
# ═══ TUNNEL PINGGY — Ne transporte que la signalisation SDP ═══
# ══════════════════════════════════════════════════════════════
def expose_with_pinggy():
    print('🌍 Création du tunnel Pinggy (signalisation WebRTC seulement)...')
    proc = subprocess.Popen(
        ['ssh', '-p', '443', '-R0:localhost:5001', '-o', 'StrictHostKeyChecking=no', '-o', 'ServerAliveInterval=30', 'a.pinggy.io'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    
    pinggy_url = None
    for line in proc.stdout:
        match = re.search(r'http[s]?://[a-zA-Z0-9.-]+\.pinggy\.(link|net)', line)
        if match:
            pinggy_url = match.group(0).replace('http://', 'https://')
            break

    if pinggy_url:
        offer_url = f'{pinggy_url}/offer'
        print('\n' + '⭐' * 30)
        print(f'🚀 WEBRTC SIGNALING URL : {offer_url}')
        print(f'🏥 HEALTH CHECK : {pinggy_url}/health')
        print('⭐' * 30)
        print()
        print('📋 POUR CONNECTER LE DASHBOARD :')
        print(f'   Copiez cette URL dans le Dashboard : {pinggy_url}')
        print()
        
        # Notifier le Dashboard via ntfy
        def notify_loop():
            while True:
                try:
                    requests.post('https://ntfy.sh/sentinelai_firas_webrtc', data=pinggy_url.encode('utf-8'))
                except:
                    pass
                time.sleep(10)
        threading.Thread(target=notify_loop, daemon=True).start()
        print('✅ Le Dashboard local va se connecter automatiquement !')
    else:
        print('⚠️ Erreur Pinggy. Vérifiez les logs.')
    
    for line in proc.stdout:
        pass

threading.Thread(target=expose_with_pinggy, daemon=True).start()

print('\n' + '═' * 60)
print('🎬 SENTINELAI V16 WebRTC — NE FERMEZ PAS CET ONGLET')
print('   Le tunnel ne transporte que la signalisation (~500 octets)')
print('   La vidéo passe directement en UDP via TURN/STUN')
print('═' * 60)

try:
    while True:
        time.sleep(5)
        print(f'\r⚡ AI FPS: {int(worker.fps)} | Frames: {worker.frame_count} | Peers WebRTC: {len(pcs)}', end='')
except KeyboardInterrupt:
    print('\n⏹️ Arrêt.')
    cam.stop()
    worker.running = False